# Day 10 — Solutions: Testing with Pytest
Functions under test and quick assert checks (full pytest suite in Markdown).

In [ ]:
def safe_divide(a: float, b: float) -> float:
    if b == 0:
        raise ZeroDivisionError('b must not be 0')
    return a / b

class InvalidEmailError(ValueError):
    pass

def validate_email(email: str) -> None:
    if '@' not in email:
        raise InvalidEmailError('Missing @ in email')

# Quick checks
assert safe_divide(10,2) == 5
assert safe_divide(-6,3) == -2
try:
    safe_divide(1,0)
except ZeroDivisionError:
    pass

validate_email('user@example.com')
try:
    validate_email('bad')
except InvalidEmailError:
    pass

Run `pytest -q` in your project to execute the full suite with parametrization and exception assertions.

---

## Exercise-by-exercise reference

Every numbered learner exercise has a matching entry here. The original
worked examples remain above; the expanded answers below add heavily
commented code, explicit reasoning, and executable checks.

### Exercise 1 — Original lesson practice

**Prompt:** Write tests for `safe_divide` and `validate_email`. **Hint:** list the contract's normal, boundary, and invalid cases before writing test code.

The earlier worked solution in this file is the reference answer. Trace it from inputs to output, then run its assertions or stated checks. The important review question is how that implementation applies the lesson contract rather than merely reproducing syntax.

### Exercise 2 — Original lesson practice

**Prompt:** Add a negative test using `pytest.raises`. **Hint:** assert the narrow exception type and, when stable, a meaningful part of its message.

The earlier worked solution in this file is the reference answer. Trace it from inputs to output, then run its assertions or stated checks. The important review question is how that implementation applies the lesson contract rather than merely reproducing syntax.

### Exercise 3 — Prediction

**Prompt:** Predict how pytest reports `assert actual == expected` compared with `assert check(actual)` when the values differ.

**Reasoning checkpoint:** Direct comparisons usually produce more useful assertion introspection. The detailed worked reasoning and commented implementation appear in the expanded solution immediately below.

### Exercise 4 — Tracing

**Prompt:** Trace fixture setup, test execution, and teardown when the test passes and when it raises.

**Reasoning checkpoint:** A yielding fixture resumes after `yield` for cleanup in both paths. The detailed worked reasoning and commented implementation appear in the expanded solution immediately below.

### Exercise 5 — Implementation

**Prompt:** Write parameterized tests for a slug function covering ordinary text, extra whitespace, punctuation, and empty text.

**Reasoning checkpoint:** Each parameter row should communicate one behavior. The detailed worked reasoning and commented implementation appear in the expanded solution immediately below.

### Exercise 6 — Debugging

**Prompt:** Repair a test whose `pytest.raises(Exception)` would accept unrelated bugs and whose protected block contains several operations.

**Reasoning checkpoint:** Assert a narrow exception around one operation. The detailed worked reasoning and commented implementation appear in the expanded solution immediately below.

### Exercise 7 — Edge case and explanation

**Prompt:** Test floating-point output and `NaN` correctly. Explain why direct equality is inappropriate for each.

**Reasoning checkpoint:** Use `pytest.approx` for tolerance and `math.isnan` for NaN. The detailed worked reasoning and commented implementation appear in the expanded solution immediately below.

## Expanded mastery lab solutions

Test observable contracts across normal, boundary, and invalid inputs. A good failure explains which behavior changed.

Read the reasoning before the code. Inline comments explain ownership, boundary choices, and why each check exists; assertions turn the stated contract into executable evidence.

### Practices 1–2 — Readable failures and reliable cleanup

Direct value comparisons let pytest display the differing values. A yielding
fixture's code after `yield` behaves like a `finally` cleanup step.

### Practices 3–5 — Focused tests


In [ ]:
import math
import re

import pytest


def slugify(text: str) -> str:
    return "-".join(re.findall(r"[a-z0-9]+", text.casefold()))


@pytest.mark.parametrize(
    ("source", "expected"),
    [
        ("Data Tools", "data-tools"),
        ("  spaced   words ", "spaced-words"),
        ("A&B!", "a-b"),
        ("", ""),
    ],
)
def test_slugify(source: str, expected: str) -> None:
    assert slugify(source) == expected


def positive_root(value: float) -> float:
    if value < 0:
        raise ValueError("value must be non-negative")
    return value ** 0.5


def test_negative_root_is_rejected() -> None:
    # Only the operation under contract is inside the context manager.
    with pytest.raises(ValueError, match="non-negative"):
        positive_root(-1)


def test_float_and_nan_semantics() -> None:
    assert 0.1 + 0.2 == pytest.approx(0.3)
    missing = float("nan")
    assert math.isnan(missing)


`NaN != NaN` by IEEE floating-point design, so equality cannot recognize it.
Tolerance checks should use a scale appropriate to the domain rather than a
randomly large allowance.
